# BayMorph — real-galaxy analysis

Configuration-driven workflow for one observed galaxy. Inspect the image-preparation products before starting the potentially expensive Bayesian fits.

## Setup

In [ ]:
from pathlib import Path
import os

from baymorph import BayMorphRunner, load_resolved_config
from baymorph.plotting.report import plot_image_preparation_report

if Path.cwd().name == "notebooks":
    os.chdir("..")

CONFIG_PATH = Path("configs/real_galaxy.yaml")
config = load_resolved_config(CONFIG_PATH)
runner = BayMorphRunner.from_config(config)

print(f"Configuration: {CONFIG_PATH}")
print(f"Object: {config['object']['name']}")
print(f"Models: {config['fit']['models']}")
print(f"Output root: {config['outputs']['root']}")

## 1. Prepare and inspect the image

BayMorph detects the target and contaminants, estimates its geometry, identifies the radial-profile plateau, builds a sky region, and derives the target-specific background and noise.

In [ ]:
image_products = runner.prepare_image(return_result=True)

print("Cutout mode:", image_products.cutout_mode)
print("Final cutout shape:", image_products.cutout.shape)
print("Cutout bounds:", image_products.cutout_bounds)
print("Target position in cutout:", (image_products.target_x, image_products.target_y))
print("Plateau radius:", image_products.plateau.radius)
print("Sky pixels:", image_products.sky_region.n_pixels)
print("Background:", image_products.background)

In [ ]:
preparation_report = (
    Path(config["outputs"]["root"])
    / config["object"]["name"]
    / "image_preparation_report.pdf"
)

plot_image_preparation_report(
    image_products,
    output_path=preparation_report,
)

print("Saved:", preparation_report)

## 2. Build or load the PSF

In [ ]:
if config["psf"]["type"] == "empirical":
    psf_candidates = runner.select_psf_candidates(return_result=True)
    psf_products = runner.build_psf(return_result=True)
    print(f"Selected {len(psf_candidates)} PSF candidates.")
else:
    psf_products = None
    print(f"Using configured {config['psf']['type']} PSF.")

## 3. Run the Bayesian fits

In [ ]:
fit_products = runner.fit(return_result=True)
fit_products

## 4. Inspect fit summaries

In [ ]:
products_by_model = (
    fit_products
    if isinstance(fit_products, dict)
    else {fit_products.model_name: fit_products}
)

for model_name, product in products_by_model.items():
    posterior = product.posterior
    logz = None if posterior is None else float(posterior.logz[-1])
    logzerr = (
        None
        if posterior is None or posterior.logzerr is None
        else float(posterior.logzerr[-1])
    )

    print(f"\n{model_name}")
    print(f"  chi2          = {product.chi2:.8g}")
    print(f"  ndof          = {product.ndof}")
    print(f"  reduced chi2  = {product.reduced_chi2:.8g}")
    print(f"  logZ          = {logz}")
    print(f"  logZ error    = {logzerr}")
    print("  best parameters:")
    for name, value in product.best_parameters.items():
        print(f"    {name:>14s}: {value:.8g}")

## 5. Generate diagnostics

In [ ]:
runner.diagnostics()